In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

In [9]:
df = pd.read_excel("/content/flight_components_condition_augmented_10000.xlsx")

In [19]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nStatistical Summary:")
print(df.describe(include='all'))

Dataset Shape: (10000, 14)

Column Names:
Index(['Component ID', 'Component Type', 'Model', 'Serial Number',
       'Manufacturer', 'Aircraft ID', 'Install Date', 'Flight Hours', 'Cycles',
       'Wear Level (%)', 'Condition Status', 'Last Inspection Date',
       'Next Inspection Due', 'Remaining Life (%)'],
      dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Component ID          10000 non-null  object 
 1   Component Type        10000 non-null  object 
 2   Model                 10000 non-null  object 
 3   Serial Number         10000 non-null  object 
 4   Manufacturer          10000 non-null  object 
 5   Aircraft ID           10000 non-null  object 
 6   Install Date          10000 non-null  object 
 7   Flight Hours          10000 non-null  float64
 8   Cycles                

In [20]:
df = df.drop(columns=[
    "Component ID",
    "Serial Number",
    "Aircraft ID",
    "Install Date",
    "Last Inspection Date",
    "Next Inspection Due"
])

In [21]:
numeric_cols = df.select_dtypes(include=np.number).columns

imputer = SimpleImputer(strategy="median")

df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

In [22]:
encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col].astype(str))

In [23]:
X = df.drop("Condition Status", axis=1)

y = df["Condition Status"]

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [26]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 98.20%


In [27]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       188
           1       0.99      1.00      0.99      1471
           2       0.00      0.00      0.00         6
           3       1.00      0.05      0.09        21
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.97      0.99      0.98       311

    accuracy                           0.98      2000
   macro avg       0.56      0.43      0.44      2000
weighted avg       0.98      0.98      0.98      2000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
print(confusion_matrix(y_test, y_pred))

[[ 186    0    0    0    0    0    2]
 [   0 1468    0    0    0    0    3]
 [   0    1    0    0    0    0    5]
 [   0   20    0    1    0    0    0]
 [   0    0    0    0    0    0    1]
 [   2    0    0    0    0    0    0]
 [   0    0    2    0    0    0  309]]


In [29]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)

print(importance)

              Feature  Importance
6  Remaining Life (%)    0.862341
5      Wear Level (%)    0.036030
0      Component Type    0.028826
1               Model    0.026539
2        Manufacturer    0.016032
3        Flight Hours    0.015485
4              Cycles    0.014747
